In [1]:
%pip install anthropic python-dotenv

from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

Looking in indexes: https://pypi-proxy.dev.databricks.com/simple

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import json


In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [3]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [6]:
dataset = generate_dataset()
print(dataset)

[{'task': "Write a Python function that takes an AWS S3 ARN string and extracts the bucket name and key path. The function should return a dictionary with 'bucket' and 'key' fields. Example ARN: 'arn:aws:s3:::my-bucket/folder/file.txt'"}, {'task': "Create a JSON object representing an AWS IAM policy that allows read-only access to a specific S3 bucket named 'company-logs'. The policy should allow s3:GetObject and s3:ListBucket actions."}, {'task': "Write a regex pattern that validates AWS EC2 instance IDs. Instance IDs start with 'i-' followed by either 8 or 17 hexadecimal characters (e.g., 'i-1234abcd' or 'i-0123456789abcdef0')."}]


In [7]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [8]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [14]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }

In [15]:
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [13]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [16]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 2.3333333333333335


In [17]:
print(json.dumps(results, indent=2))

[
  {
    "output": "I'll help you write a Python function to extract the bucket name and key path from an AWS S3 ARN string.\n\n```python\ndef parse_s3_arn(arn):\n    \"\"\"\n    Extracts bucket name and key path from an AWS S3 ARN string.\n    \n    Args:\n        arn (str): AWS S3 ARN string (e.g., 'arn:aws:s3:::my-bucket/folder/file.txt')\n    \n    Returns:\n        dict: Dictionary with 'bucket' and 'key' fields\n        \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    # Validate basic ARN structure\n    if not arn or not isinstance(arn, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    # Check if it starts with the S3 ARN prefix\n    if not arn.startswith('arn:aws:s3:::'):\n        raise ValueError(\"Invalid S3 ARN format. Must start with 'arn:aws:s3:::'\")\n    \n    # Remove the ARN prefix\n    s3_path = arn[len('arn:aws:s3:::'):]\n    \n    # Split bucket and key\n    if '/' in s3_path:\n        parts = s3_path.spli